# 🧹 Phase 1: Data Cleaning & Pre-processing

Based on the initial analysis, this notebook implements a **conservative cleaning strategy**.
We aim to fix structural errors (missing IDs) and optimize data types while preserving as much information as possible (e.g., keeping anime with partial data).

**Key Actions:**
1.  **Impute** missing values where logic allows (e.g., `favorites=0`, `is_rewatching=False`).
2.  **Drop** only structurally corrupted rows (missing Primary/Foreign Keys).
3.  **Extract** useful features (e.g., `year` from `start_date`).
4.  **Optimize** data types (float64 -> float32/16, objects -> categories) to reduce memory usage.
5.  **Save** to optimized Parquet format for the next steps.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import gc
from pathlib import Path
from tqdm.notebook import tqdm

# 🧪 Standard Import (requires 'solution' package installed)
from solution.src.common import RAW_DATA_DIR, PROCESSED_DATA_DIR

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print(f"Reading from: {RAW_DATA_DIR}")
print(f"Saving to:    {PROCESSED_DATA_DIR}")

In [ ]:
def get_memory_usage_mb(df: pd.DataFrame) -> float:
    """Returns the memory usage of the DataFrame in MB."""
    return df.memory_usage(deep=True).sum() / 1024 ** 2

def optimize_dtypes(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """
    Downcast numeric types and convert low-cardinality objects to categories.
    Reports memory reduction.
    """
    start_mem = get_memory_usage_mb(df)
    
    for col in df.columns:
        col_type = df[col].dtype

        # Numeric Downcasting
        if pd.api.types.is_numeric_dtype(col_type):
            if pd.api.types.is_integer_dtype(col_type):
                # Check range for integers
                min_val, max_val = df[col].min(), df[col].max()
                if min_val >= 0:
                    if max_val <= 255: df[col] = df[col].astype(np.uint8)
                    elif max_val <= 65535: df[col] = df[col].astype(np.uint16)
                    elif max_val <= 4294967295: df[col] = df[col].astype(np.uint32)
                else:
                    if min_val >= -128 and max_val <= 127: df[col] = df[col].astype(np.int8)
                    elif min_val >= -32768 and max_val <= 32767: df[col] = df[col].astype(np.int16)
                    elif min_val >= -2147483648 and max_val <= 2147483647: df[col] = df[col].astype(np.int32)

            elif pd.api.types.is_float_dtype(col_type):
                # We can often use float32 instead of float64 for our ranges
                # Skip if column has NaN and needs float behavior, but float32 handles NaNs too.
                df[col] = df[col].astype(np.float32)

        # Categorical Conversion (simple heuristic: < 50% unique values)
        elif pd.api.types.is_object_dtype(col_type):
            num_unique = len(df[col].unique())
            num_total = len(df)
            if num_total > 100 and (num_unique / num_total < 0.5):
                df[col] = df[col].astype('category')
    
    end_mem = get_memory_usage_mb(df)
    if verbose:
        reduction = 100 * (start_mem - end_mem) / start_mem
        print(f"Memory optimization: {start_mem:.2f} MB -> {end_mem:.2f} MB ({reduction:.1f}% reduction)")

    return df

def save_refined(df: pd.DataFrame, filename: str):
    """Saves DataFrame to Parquet (preferred) or CSV using compression and performs GC."""
    file_path = PROCESSED_DATA_DIR / f"{filename.replace('.csv', '.parquet')}"
    print(f"Saving {filename} to {file_path}...")
    try:
        # Use zstd compression for good balance of speed and size
        df.to_parquet(file_path, index=False, compression='zstd')
        print("Done.")
    except Exception as e:
        print(f"Error saving parquet: {e}. Falling back to CSV.")
        df.to_csv(PROCESSED_DATA_DIR / filename, index=False)
    
    # Force Garbage Collection to free memory immediately after save
    gc.collect()

## 1. Characters (`characters.csv`)
- Drop rows with missing `character_mal_id`.
- Fill missing `favorites` with 0.
- Drop `about` column (mostly empty/text heavy text not needed for general stats).

In [ ]:
df_chars = pd.read_csv(RAW_DATA_DIR / "characters.csv")
initial_rows = len(df_chars)

# Stats before cleaning
missing_ids = df_chars['character_mal_id'].isna().sum()
missing_favs = df_chars['favorites'].isna().sum()

# 1. Drop missing IDs
df_chars = df_chars.dropna(subset=['character_mal_id'])
rows_after_drop = len(df_chars)

# 2. Fill favorites
df_chars['favorites'] = df_chars['favorites'].fillna(0)

# 3. Drop 'about'
dropped_cols = []
if 'about' in df_chars.columns:
    df_chars = df_chars.drop(columns=['about'])
    dropped_cols.append('about')

# Report
print(f"--- Characters Cleaning Report ---")
print(f"Initial Rows: {initial_rows}")
print(f"Rows Dropped (Missing ID): {initial_rows - rows_after_drop}")
print(f"Favorites Imputed (NaN -> 0): {missing_favs}")
print(f"Columns Dropped: {dropped_cols}")
print(f"Final Shape: {df_chars.shape}")

# Optimize
df_chars = optimize_dtypes(df_chars)
save_refined(df_chars, "characters.csv")

## 2. Anime Details (`details.csv`)
- **Year**: Derive from `start_date` where missing.
- **Dates**: Convert to datetime objects.
- **Missing Stats**: Keep `score`, `rank` as NaN (preserve "unranked" status).
- **Columns**: Keep `demographics` and others even if they look empty (might have sparse data).

In [ ]:
df_details = pd.read_csv(RAW_DATA_DIR / "details.csv")
initial_rows = len(df_details)

# 1. Date Parsing
df_details['start_date'] = pd.to_datetime(df_details['start_date'],  errors='coerce')
df_details['end_date'] = pd.to_datetime(df_details['end_date'],  errors='coerce')

# 2. Impute Year from Start Date
# If year is missing, try to get it from start_date
missing_year_before = df_details['year'].isna().sum()
df_details['year'] = df_details['year'].fillna(df_details['start_date'].dt.year) # type: ignore
missing_year_after = df_details['year'].isna().sum()

# 3. Drop useless columns if verified empty (heuristic check)
# We keep them if they have valid info, but analysis showed some might be empty lists.
# However, we'll keep them to be conservative as requested.

# Report
print(f"--- Details Cleaning Report ---")
print(f"Initial Rows: {initial_rows}")
print(f"Year Imputed from Start Date: {missing_year_before - missing_year_after}")
print(f"Remaining Missing Years: {missing_year_after}")
print(f"Final Shape: {df_details.shape}")

# Optimize
df_details = optimize_dtypes(df_details)

# Explicitly convert known low-cardinality columns to category (based on analysis)
cat_cols = ['type', 'status', 'source', 'rating', 'season']
for col in cat_cols:
    if col in df_details.columns:
        df_details[col] = df_details[col].astype('category')

save_refined(df_details, "details.csv")

## 3. Anime Stats (`stats.csv`)
- **Votes**: Fill NaN score votes with 0.
- **Percentages**: Fill NaN percentages with 0.
- **Conserve Rows**: Do not drop rows just because they have no votes.

In [ ]:
df_stats = pd.read_csv(RAW_DATA_DIR / "stats.csv")
initial_rows = len(df_stats)

# Fill score columns with 0
score_cols = df_stats.filter(like='score_').columns
nan_in_scores = df_stats[score_cols].isna().sum().sum()
df_stats[score_cols] = df_stats[score_cols].fillna(0)

# Report
print(f"--- Stats Cleaning Report ---")
print(f"Initial Rows: {initial_rows}")
print(f"Total NaN values filled in score columns: {nan_in_scores}")
print(f"Final Shape: {df_stats.shape}")

# Optimize
df_stats = optimize_dtypes(df_stats)
save_refined(df_stats, "stats.csv")

## 4. User Ratings (`ratings.csv`)
- **Huge File**: Process carefully.
- **Username**: Drop NaN (orphan votes).
- **Rewatching**: Assume `False` (0) if missing.

In [ ]:
# Accessing the large file with Chunking
chunk_size = 5_000_000
chunks = []
processed_count = 0

# Metrics
total_rows_read = 0
total_dropped_username = 0
total_rewatch_imputed = 0

csv_path = RAW_DATA_DIR / "ratings.csv"
print(f"Processing {csv_path} in chunks...")

# Estimated total chunks (rough calculation based on 125M rows known from analysis or file size)
# ~125M rows / 5M chunk = 25 chunks
ESTIMATED_CHUNKS = 25 

with pd.read_csv(csv_path, chunksize=chunk_size) as reader:
    for chunk in tqdm(reader, total=ESTIMATED_CHUNKS, desc="Processing chunks"):
        rows_in_chunk = len(chunk)
        total_rows_read += rows_in_chunk

        # 1. Drop missing username
        missing_user_mask = chunk['username'].isna()
        dropped_count = missing_user_mask.sum()
        total_dropped_username += dropped_count

        chunk = chunk.dropna(subset=['username'])

        # 1.5. Score Sanity Check
        # Ensure scores are >= 0 (handle potential -1 or corrupted negative values)
        chunk = chunk[(chunk['score'] >= 0) & (chunk['score'] <= 10)]

        # 2. Fill rewatching
        missing_rewatch_mask = chunk['is_rewatching'].isna()
        total_rewatch_imputed += missing_rewatch_mask.sum()

        chunk['is_rewatching'] = chunk['is_rewatching'].fillna(0).astype(np.uint8) # 0 or 1

        # 3. Optimize types strictly to save memory
        chunk['anime_id'] = chunk['anime_id'].astype(np.int32)
        chunk['score'] = chunk['score'].astype(np.int8)
        chunk['num_watched_episodes'] = chunk['num_watched_episodes'].astype(np.int32)
        chunk['status'] = chunk['status'].astype('category')

        chunks.append(chunk)
        processed_count += len(chunk)

# Concatenate all
print("Concatenation...")
df_ratings = pd.concat(chunks, ignore_index=True)
print("Concatenation complete.")

print(f"--- Ratings Cleaning Report ---")
print(f"Total Rows Read: {total_rows_read}")
print(f"Rows Dropped (Missing Username): {total_dropped_username}")
print(f"Rewatching Flags Imputed: {total_rewatch_imputed}")
print(f"Final Shape: {df_ratings.shape}")

# Note: optimize_dtypes might be slow on full DF, but safe here if memory holds.
# Since we manually optimized chunks, we can skip or run it for final check.
# Let's run it to catch any non-optimized cols and get final memory stats.
df_ratings = optimize_dtypes(df_ratings)

save_refined(df_ratings, "ratings.csv")

# Free memory
del df_ratings, chunks
gc.collect()

## 5. Other Files
Processing the remaining files with standard optimizations:
- `person_details.csv`
- `profiles.csv`
- `favs.csv`
- `recommendations.csv`
- Relational tables (`*_works.csv`, `*_nicknames.csv`)

In [ ]:
remaining_files = [
    "person_details.csv",
    "profiles.csv",
    "favs.csv",
    "recommendations.csv",
    "character_anime_works.csv",
    "character_nicknames.csv",
    "person_alternate_names.csv",
    "person_anime_works.csv",
    "person_voice_works.csv"
]

for filename in tqdm(remaining_files, desc="Processing other files"):
    file_path = RAW_DATA_DIR / filename
    if os.path.exists(file_path):
        # Using print inside tqdm loop might break the bar visual, but useful for logs
        # tqdm.write() is better
        tqdm.write(f"Processing {filename}...")
        df = pd.read_csv(file_path)
        initial_rows = len(df)

        # Specific fixes
        if filename == "person_details.csv":
            # We let birthday be NaN (datetime conversion handles errors)
            # Note: Analysis showed ~77.5% missing birthdays, so this column is mostly informative.
            df['birthday'] = pd.to_datetime(df['birthday'], errors='coerce')

        if filename == "profiles.csv":
            # Date columns
            for date_col in ['joined']:
                if date_col in df.columns:
                    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

        df = optimize_dtypes(df, verbose=False) # Suppress detailed print for cleaner output loop
        
        # Manually print brief stats
        final_rows = len(df)
        tqdm.write(f" -> Processed {filename}. Rows: {initial_rows} -> {final_rows}")
        
        save_refined(df, filename)